# AIDEN — Multimodal Inference on Google Colab (Free GPU)

This notebook loads **LLaVA 1.6 Mistral 7B** on a free T4 GPU and exposes a public API endpoint via **ngrok**.
Your local AIDEN backend will proxy multimodal requests here automatically.

⏱️ **Setup time:** ~5 minutes
🚀 **Once running:** Works until Colab runtime disconnects

---
## Step 1: Open Colab with GPU

1. Go to [Google Colab](https://colab.research.google.com).
2. Click **File → New Notebook**.
3. In the top menu, click **Runtime → Change runtime type**.
4. Select **T4 GPU** as the hardware accelerator.
5. Click **Save**.

---
## Step 2: Install Dependencies & Load Model

Run this entire cell (copy-paste):

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes peft pillow requests fastapi uvicorn pyngrok nest-asyncio

import torch
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
from PIL import Image
import base64
from io import BytesIO
import nest_asyncio
nest_asyncio.apply()

print("Loading LLaVA 1.6 Mistral 7B with 4-bit quantization...")

model_id = "llava-hf/llava-v1.6-mistral-7b-hf"
processor = LlavaNextProcessor.from_pretrained(model_id)

model = LlavaNextForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    load_in_4bit=True,
    device_map="auto"
)

print("Model loaded successfully!")
print(f"   Device: {model.device}")
print(f"   VRAM used: ~4 GB (4-bit quantization)")

**Expected output:**
```
Loading LLaVA 1.6 Mistral 7B with 4-bit quantization...
Loading checkpoint shards: 100%|██████████| 4/4 [00:15<00:00, 3.8s/it]
Model loaded successfully!
   Device: cuda:0
   VRAM used: ~4 GB (4-bit quantization)
```

---
## Step 3: Set Up ngrok (Authentication Required)

Run this cell (you need a free ngrok auth token):

In [ ]:
from pyngrok import ngrok

# Option A: Set your token directly (replace with your actual token)
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"  # Get from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Option B: Use Colab Secrets (recommended for sharing notebooks)
# from google.colab import userdata
# ngrok.set_auth_token(userdata.get('ngrok_token'))

print("ngrok authenticated")

**How to get your token:**
1. Sign up at [ngrok.com](https://ngrok.com) (free).
2. Go to [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).
3. Copy your token and paste it into the cell above.

---
## Step 4: Quick Inference Test (Optional)

Run this to verify the model is working before starting the server:

In [ ]:
# Test with a simple dummy image
from PIL import Image
import torch

# Create a dummy image (black square)
dummy_image = Image.new('RGB', (224, 224), color='black')

conversation = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "What do you see?"}
    ]}
]

inputs = processor.apply_chat_template(
    conversation,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True
)
# transformers >=4.46 returns a Tensor directly, older versions return BatchFeature (dict-like)
if isinstance(inputs, dict) or hasattr(inputs, 'items'):
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
else:
    inputs = inputs.to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50, temperature=0.7)

response = processor.decode(outputs[0], skip_special_tokens=True)
print("Test response:", response)

If you see a response (even "I see a black image"), the model is working.

---
## Step 5: Create the FastAPI Server

Run this cell (starts the server + ngrok tunnel):

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import uvicorn
import asyncio
from datetime import datetime

app = FastAPI(title="AIDEN Multimodal Proxy", version="1.0.0")

@app.get("/status")
async def status():
    """Check if the server is alive."""
    return {
        "status": "ok",
        "model": model_id,
        "device": str(model.device),
        "timestamp": datetime.utcnow().isoformat(),
    }

@app.post("/analyze")
async def analyze(
    file: UploadFile = File(...),
    prompt: str = Form("Describe this pipeline diagram.")
):
    """
    Analyze a pipeline diagram image.
    """
    try:
        # 1. Read and decode image
        img_bytes = await file.read()
        if len(img_bytes) == 0:
            raise HTTPException(status_code=400, detail="Empty image file")
        
        image = Image.open(BytesIO(img_bytes))
        
        # 2. Prepare conversation
        conversation = [
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]}
        ]
        
        # 3. Tokenize
        inputs = processor.apply_chat_template(
            conversation,
            tokenize=True,
            return_tensors="pt",
            add_generation_prompt=True
        )
        # transformers >=4.46 returns a Tensor directly, older versions return BatchFeature (dict-like)
        if isinstance(inputs, dict) or hasattr(inputs, 'items'):
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
        else:
            inputs = inputs.to(model.device)
        
        # 4. Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                pad_token_id=processor.tokenizer.eos_token_id,
            )
        
        # 5. Decode
        response = processor.decode(outputs[0], skip_special_tokens=True)
        
        # 6. Extract assistant reply
        if "assistant" in response:
            response = response.split("assistant")[-1].strip()
        
        return {
            "success": True,
            "analysis": response,
            "model": model_id,
            "prompt": prompt,
            "tokens": len(response.split()),
            "timestamp": datetime.utcnow().isoformat(),
        }
    
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"success": False, "error": str(e)}
        )

# Start ngrok
public_url = ngrok.connect(8000)
print(f"\nPublic URL: {public_url}")
print(f"   Health check: {public_url}/status")
print(f"   Analyze endpoint: {public_url}/analyze")

# Start uvicorn in background
import threading
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server is running (in background)")
print("   Keep this notebook open – closing it stops the service.")

After running, copy the **Public URL** – you'll need it for your local backend.

---
## Step 6: Verify the Server Is Responding

In Colab, run:

In [ ]:
import requests
import json

url = "https://your-ngrok-url.ngrok-free.app"  # Replace with your URL
response = requests.get(f"{url}/status")
print(json.dumps(response.json(), indent=2))

**Expected output:**
```json
{
  "status": "ok",
  "model": "llava-hf/llava-v1.6-mistral-7b-hf",
  "device": "cuda:0",
  "timestamp": "2026-07-26T10:00:00.000000"
}
```

---
## Step 7: Configure AIDEN Backend to Use Colab

### 7.1 – Update backend/.env:

```env
# Multimodal AI — Remote Proxy Mode
MULTIMODAL_REMOTE_URL=https://your-ngrok-url.ngrok-free.app
# MULTIMODAL_ENABLED=False   # Optional: disabled local loading
```

### 7.2 – Ensure the service uses remote mode

Your `backend/app/services/multimodal_service.py` already has the proxy logic. If not, here's the complete file:

```python
import httpx
import base64
import logging
from typing import Optional, Dict, Any
from app.config import settings

logger = logging.getLogger(__name__)

class MultimodalService:
    """Multimodal service – can run locally (GPU) or proxy to Colab."""

    def __init__(self):
        self.remote_url = settings.MULTIMODAL_REMOTE_URL
        self._local_loaded = False
        self._model = None
        self._processor = None

        # If remote URL is set, we skip local loading entirely
        if self.remote_url:
            logger.info(f"Multimodal in remote proxy mode: {self.remote_url}")
        else:
            logger.info("Multimodal in local mode (fallback)")

    def is_available(self) -> bool:
        """Check if the service is available (remote or local)."""
        if self.remote_url:
            return True  # Assume available – health check will verify
        return self._local_loaded

    async def analyze_diagram(
        self,
        image_data: str,
        prompt: Optional[str] = None,
        temperature: float = 0.7,
        max_tokens: int = 512,
    ) -> Dict[str, Any]:
        """Analyze a diagram – routes to remote or local."""
        if self.remote_url:
            return await self._analyze_remote(image_data, prompt, temperature, max_tokens)
        return await self._analyze_local(image_data, prompt, temperature, max_tokens)

    async def _analyze_remote(
        self,
        image_data: str,
        prompt: Optional[str],
        temperature: float,
        max_tokens: int,
    ) -> Dict[str, Any]:
        """Proxy request to Colab."""
        try:
            # Decode base64 to bytes
            if image_data.startswith("data:image"):
                image_data = image_data.split(",")[1]
            img_bytes = base64.b64decode(image_data)

            async with httpx.AsyncClient(timeout=120.0) as client:
                response = await client.post(
                    f"{self.remote_url}/analyze",
                    files={"file": ("image.png", img_bytes, "image/png")},
                    data={"prompt": prompt or "Describe this pipeline diagram."},
                )

                if response.status_code == 200:
                    data = response.json()
                    return {
                        "success": True,
                        "analysis": data.get("analysis", "No analysis returned."),
                        "model": data.get("model", self.remote_url),
                        "tokens": data.get("tokens", 0),
                        "mode": "remote",
                    }
                else:
                    return {
                        "success": False,
                        "error": f"Colab returned {response.status_code}: {response.text[:200]}",
                    }

        except httpx.TimeoutException:
            return {"success": False, "error": "Colab inference timeout (120s)."}
        except Exception as e:
            return {"success": False, "error": str(e)}

    async def _analyze_local(self, image_data: str, prompt: Optional[str], temperature: float, max_tokens: int):
        """Original local implementation – kept for fallback."""
        # ... (your existing local code) ...
        return {"success": False, "error": "Local mode not implemented in this example."}

    async def health_check(self) -> Dict[str, Any]:
        """Check remote health if in remote mode."""
        if not self.remote_url:
            return {"mode": "local", "available": self._local_loaded}

        try:
            async with httpx.AsyncClient(timeout=5.0) as client:
                response = await client.get(f"{self.remote_url}/status")
                if response.status_code == 200:
                    return {
                        "mode": "remote",
                        "remote_url": self.remote_url,
                        "status": "ok",
                        "details": response.json(),
                    }
                else:
                    return {"mode": "remote", "remote_url": self.remote_url, "status": "error", "http_status": response.status_code}
        except Exception as e:
            return {"mode": "remote", "remote_url": self.remote_url, "status": "error", "detail": str(e)}

# Singleton
multimodal_service = MultimodalService()
```

### 7.3 – Update the /multimodal/status endpoint

In `backend/app/api/v1/multimodal.py`:

```python
@router.get("/status")
async def multimodal_status(current_user: User = Depends(get_current_user)):
    """Check multimodal service availability."""
    health = await multimodal_service.health_check()
    return {
        "available": multimodal_service.is_available(),
        "mode": health.get("mode", "unknown"),
        "remote_url": health.get("remote_url"),
        "remote_health": health.get("status", "unknown"),
        "local_loaded": multimodal_service.is_available() and not multimodal_service.remote_url,
    }
```

---
## Step 8: Test the Full Flow

### 8.1 – Restart your local backend

```bash
cd D:\aiden\backend
uvicorn app.main:app --reload --port 8000
```

### 8.2 – Check the status endpoint

```bash
curl http://localhost:8000/api/v1/multimodal/status \
  -H "Authorization: Bearer YOUR_TOKEN"
```

**Expected output:**
```json
{
  "available": true,
  "mode": "remote",
  "remote_url": "https://xxxx.ngrok-free.app",
  "remote_health": "ok",
  "local_loaded": false
}
```

### 8.3 – Upload a diagram via the frontend

1. Start the frontend: `cd frontend && npm run dev`
2. Open http://localhost:5174/multimodal
3. Upload one of your generated diagrams (e.g., `diagrams/diagram_0000.png`)
4. Click **Analyze**
5. You should receive a detailed description within 5–15 seconds

---
## Step 9: Keep the Colab Session Alive

| Action | Why |
|--------|-----|
| Keep the notebook open | If you close it, the server stops. |
| Click the "Connect" button occasionally | Prevents idle disconnection. |
| Use a "keep-alive" script | Optional: send a ping every 5 minutes. |
| Upgrade to Colab Pro | Longer sessions and better GPUs (~$10/month). |

---
## Troubleshooting

| Issue | Likely Cause | Fix |
|-------|--------------|-----|
| ngrok authentication error | Token not set or invalid | Replace `YOUR_NGROK_AUTH_TOKEN` with your actual token from dashboard. |
| ModuleNotFoundError | Missing package | Re-run the installation cell. |
| Out of memory (CUDA OOM) | 4-bit not applied or image too large | Check `load_in_4bit=True`. Reduce image size to < 800×600. |
| Timeout error | Colab server slow or disconnected | Increase timeout to 120s; check Colab logs. |
| 403 Forbidden from ngrok | ngrok auth token expired | Update your token in the notebook. |
| Server stops after a few hours | Colab idle timeout | You need to re-run the notebook. Use `!nohup` or keep the tab active. |

---
## Summary

| Step | Status |
|------|--------|
| Colab notebook created | ✅ |
| LLaVA loaded with 4-bit | ✅ |
| FastAPI server running | ✅ |
| ngrok tunnel active | ✅ |
| Backend remote proxy configured | ✅ |
| Frontend ready to upload | ✅ |

---
*Notebook generated for AIDEN — AI Data Engineering*